<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/03_runtime.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap03) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap03-runtime.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 2](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/02_herramientas.ipynb) · ➡️ [Cap. 4](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/04_memoria_corto_plazo.ipynb)

# Capítulo 3 — Del portátil a la nube

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** El equipo web de TiendaNova pregunta: *«¿A qué URL llamamos?»*. El agente vive en tu portátil.

| | |
|---|---|
| **🧠 Componente agéntico** | Entorno de ejecución: el agente como servicio |
| **La necesidad** | Que el agente esté disponible para miles de clientes, aislado entre ellos y sin gestionar servidores |
| **Lo que construimos** | Proyecto con la AgentCore CLI y despliegue en **Runtime** |
| **🏠 En local** | ☁️ No aplica: este componente existe porque los clientes están en Internet. |
| **☁️ Por qué AWS al salir a Internet** | Un proceso local es un único punto de fallo: todos los usuarios comparten proceso (y lo que haya en memoria), no escala y muere si cierras el portátil. Runtime ejecuta cada sesión en su propia **microVM**, que conserva su estado mientras la sesión está viva y se apaga cuando deja de usarse; escala de cero a miles y cobra por uso. |
| **En el ordenador sería…** | **Encender el ordenador para todos**: una máquina disponible 24/7, con cada usuario en su propia sesión aislada |
| **Requisitos del caso** | Base de todos los demás |
| **Prerrequisito · duración** | Capítulo 2 · 60 min |
| **⚠️ Dónde se rompe la analogía** | Más que «un ordenador para todos», Runtime da **un ordenador a cada sesión**, que se apaga tras 15 minutos sin uso (como máximo vive 8 horas): el aislamiento es la mitad de su valor, y lo que guardes dentro dura lo que dure ese ordenador. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=3)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(3)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(3)
# ponerse_al_dia(3)   # ← descomenta si llegas tarde o algo se rompió

## 1. Experimento de rotura: un único agente para todos los usuarios

El error clásico al montar un servidor: crear **un** agente global y reutilizarlo para todas las peticiones. Dos clientes escriben seguidos al mismo proceso. *(Datos ficticios.)*

> ✋ **Predice antes de ejecutar.** El usuario B no dice quién es: solo pide que le confirmen la dirección de entrega. ¿Qué responderá el agente? ¿Y qué viaja en la petición de B que B no ha escrito?

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agente_global = Agent(model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0.2),
                      system_prompt="Eres el asistente de TiendaNova.", callback_handler=None)
print("Usuario A:", agente_global("Soy Laura García, vivo en la calle Mayor 3 de Valencia y mi teléfono es 600 123 456. "
                                  "Apunta que prefiero las entregas por la tarde."))
print("\nUsuario B:", agente_global("Hola, ¿me confirmas la dirección y el teléfono que tenéis para mi entrega?"))

In [ ]:
# Comprobación determinista: ¿qué contexto acompaña a la petición de B?
import json
historial = json.dumps(agente_global.messages, ensure_ascii=False)
print("Mensajes en el contexto que se envía con la petición de B:", len(agente_global.messages))
print("¿Va el teléfono de Laura dentro?", "600 123 456" in historial)
print("¿Lo ha repetido el agente en su respuesta a B?", "600 123 456" in str(agente_global.messages[-1]))

Desde el punto de vista del modelo, B es la misma persona que A: es **la misma conversación**. Lo más probable es que le confirme la dirección y el teléfono de Laura; y aunque esta vez no lo hiciera, la comprobación muestra que los datos de Laura **viajan en el contexto** de cada petición de B. No es un fallo del modelo: es un fallo de **arquitectura**.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**Encender el ordenador para todos**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap03-runtime.html). Haz llegar clientes al agente en tu máquina y en Runtime, y mira qué comparten, qué conserva cada sesión y hasta cuándo. Vuelve cuando sepas responder a esta pregunta: **¿qué tiene que tener cada sesión para que dos clientes no compartan nunca nada, y qué conserva la microVM de una sesión y durante cuánto tiempo?**

**En dos líneas:** un agente en producción es un **servicio** que atiende a muchas personas a la vez, y cada **sesión** debe estar aislada de las demás. Runtime da a cada `runtimeSessionId` su propia microVM: lo que el código guarde en memoria lo ve solo esa sesión y dura lo que viva la microVM (hasta 15 minutos sin actividad, 8 horas como máximo, o hasta el siguiente despliegue). Lo que tiene que durar más, como la conversación de mañana, vive fuera (capítulo 4). Este componente no existe en local: aparece porque los clientes están en Internet.

**Fuga de datos personales.** La solución tiene dos partes:
1. **No compartir objetos entre peticiones:** nuestro `main.py` creará un `Agent` nuevo en cada invocación.
2. **Aislar cada sesión a nivel de cómputo:** Runtime ejecuta cada `runtimeSessionId` en su propia microVM. Aunque alguien deje un objeto global, una sesión nunca ve la memoria de otra.

## 2. El proyecto de la AgentCore CLI

In [ ]:
if PROJECT.exists():
    print("El proyecto ya existe:", PROJECT)
else:
    cli("agentcore create --project-name NovaAssist --name NovaAgent --language Python --framework Strands "
        "--model-provider Bedrock --memory none --build CodeZip", cwd=LAB_ROOT)

In [ ]:
arbol(PROJECT)   # en lugar de `find`, que en Windows es otro comando

> Si la CLI pide confirmaciones interactivas, ejecuta el mismo comando en una terminal desde la carpeta `labs/`. Las *flags* cambian entre versiones: `agentcore create --help`.

## 3. El código del agente

- `settings.py` se **genera** desde `lab_state.json` con `write_settings()`.
- Los pedidos se empaquetan con el agente como módulo Python. Las herramientas del capítulo 2 pasan a `herramientas.py`.
- `respuesta.py` devuelve, junto a la respuesta, las herramientas usadas, los *tokens* y una **huella de la microVM**: un identificador que se genera una vez por proceso y un contador de invocaciones. Con ella veremos qué microVM atiende cada mensaje.

In [ ]:
write_settings()
# Versiones probadas juntas (octubre 2026): strands-agents >= 1.57 rompe la integración de Memory de bedrock-agentcore 1.22
fijar_dependencia(VERSIONES["strands-agents"])
fijar_dependencia(VERSIONES["bedrock-agentcore"])

In [ ]:
empaquetar_pedidos()

In [ ]:
%%writefile NovaAssist/app/NovaAgent/herramientas.py
"""Herramientas locales del capítulo 2 sobre el dataset sintético (en el capítulo 7 pasan a una Lambda sobre Iceberg)."""
import json

from strands import tool

from datos_pedidos import PEDIDOS


def _publico(p: dict) -> dict:
    datos = {k: v for k, v in p.items() if k != "lineas_json"}
    datos["lineas"] = json.loads(p["lineas_json"])
    return datos


@tool
def consultar_pedido(pedido_id: str) -> dict:
    """Devuelve estado, fechas, importes (en céntimos) y líneas de un pedido de TiendaNova.

    Args:
        pedido_id: Identificador del pedido con formato TN-NNNN.
    """
    pedido = PEDIDOS.get(pedido_id.upper())
    return _publico(pedido) if pedido else {"error": f"No existe el pedido {pedido_id}"}


@tool
def consultar_envio(pedido_id: str) -> dict:
    """Devuelve transportista, estado y fechas estimada y real de entrega de un pedido.

    Args:
        pedido_id: Identificador del pedido con formato TN-NNNN.
    """
    p = PEDIDOS.get(pedido_id.upper())
    if not p:
        return {"error": f"No existe el pedido {pedido_id}"}
    return {k: p[k] for k in ("transportista", "estado", "eta", "fecha_entrega")}


@tool
def solicitar_devolucion(pedido_id: str, motivo: str) -> dict:
    """Abre una solicitud de devolución (RMA) para un pedido entregado.

    Args:
        pedido_id: Identificador del pedido.
        motivo: Motivo de la devolución indicado por el cliente.
    """
    return {"rma": f"RMA-{pedido_id.upper()}", "estado": "ABIERTA", "motivo": motivo}


@tool
def procesar_reembolso(pedido_id: str, importe_centimos: int, motivo: str) -> dict:
    """Emite un reembolso al método de pago original. Acción con impacto económico.

    Args:
        pedido_id: Identificador del pedido.
        importe_centimos: Importe a reembolsar en céntimos (entero).
        motivo: Justificación del reembolso.
    """
    return {"reembolso_id": f"RF-{pedido_id.upper()}", "importe_centimos": importe_centimos, "estado": "EMITIDO"}


@tool
def escalar_a_humano(resumen: str, prioridad: str = "normal") -> dict:
    """Crea un ticket para el equipo humano de soporte con un resumen del caso.

    Args:
        resumen: Resumen del caso, qué se ha intentado y qué falta.
        prioridad: 'baja', 'normal' o 'alta'.
    """
    return {"ticket": "SUP-0001", "prioridad": prioridad, "estado": "EN_COLA"}


HERRAMIENTAS = [consultar_pedido, consultar_envio, solicitar_devolucion, procesar_reembolso, escalar_a_humano]

In [ ]:
%%writefile NovaAssist/app/NovaAgent/respuesta.py
"""Lo que el agente devuelve a los notebooks: respuesta, herramientas, tokens y huella de la microVM."""
import os
import time

# Se ejecuta UNA vez por proceso. En Runtime, un proceso = una microVM = una sesión.
_MICROVM = {"id": os.urandom(3).hex(), "arranque": time.time(), "invocaciones": 0}


def huella() -> dict:
    """Identifica la microVM que atiende la invocación: misma sesión viva → mismo id y el contador sube."""
    _MICROVM["invocaciones"] += 1
    return {"id": _MICROVM["id"], "invocacion": _MICROVM["invocaciones"],
            "segundos_viva": int(time.time() - _MICROVM["arranque"])}


def respuesta(result, **extra) -> dict:
    """Respuesta del agente + herramientas usadas + tokens + huella (para la verificación en los notebooks)."""
    try:
        herramientas = sorted(result.metrics.tool_metrics.keys())
        uso = dict(result.metrics.accumulated_usage)
    except Exception:  # noqa: BLE001
        herramientas, uso = [], {}
    salida = {"respuesta": str(result), "herramientas": herramientas, "tokens": uso, "microvm": huella()}
    if getattr(result, "stop_reason", None) == "guardrail_intervened":
        salida["guardrail"] = True
    salida.update({k: v for k, v in extra.items() if v is not None})
    return salida

In [ ]:
%%writefile NovaAssist/app/NovaAgent/prompts.py
SYSTEM_PROMPT = """Eres Nova Assist, el asistente de postventa de TiendaNova.
- En tu primer mensaje te presentas como asistente de inteligencia artificial.
- Respondes en el idioma del cliente, con tono cercano y profesional.
- Si no dispones de un dato, lo dices claramente: nunca inventes estados de pedido, fechas ni importes."""

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel

from herramientas import HERRAMIENTAS
from prompts import SYSTEM_PROMPT
from respuesta import respuesta
from settings import MODEL_ID, REGION

app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload, context):
    # Un Agent NUEVO en cada invocación: no arrastra el historial de nadie… tampoco el de este cliente.
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2),
        system_prompt=SYSTEM_PROMPT,
        tools=HERRAMIENTAS,
        callback_handler=None,
    )
    return respuesta(agent(payload.get("prompt", "")))


if __name__ == "__main__":
    app.run()

## 4. Prueba local rápida (sin desplegar)

Dos clientes distintos, dos sesiones distintas, en tu máquina. Fíjate en la línea 🖥️.

In [ ]:
probar_local("¿Dónde está mi pedido TN-1001?", session_id=new_session_id())
probar_local("¿Qué artículos tiene el pedido TN-1003?", session_id=new_session_id())

Las dos sesiones tienen **la misma huella**: en tu máquina, todas las sesiones comparten un proceso. Es lo que hacía posible la fuga del experimento 1.

> Para el ciclo completo de desarrollo, ejecuta `agentcore dev` **en una terminal** dentro de `labs/NovaAssist`: abre el **Agent Inspector** en el navegador.

## 5. Desplegar

In [ ]:
cli('agentcore deploy --dry-run')

In [ ]:
# El primer despliegue hace el bootstrap de CDK (unos minutos). --yes aprueba el bootstrap sin preguntar.
cli('agentcore deploy --yes')

In [ ]:
cli('agentcore status')

In [ ]:
arn = buscar_runtime("NovaAgent")
if arn:
    state_set(AGENT_ARN=arn)
# Si no lo encuentra: state_set(AGENT_ARN="<ARN de agentcore status>")

## 6. Invocar el agente desplegado

In [ ]:
invocar("¿Dónde está mi pedido TN-1001?")

## 7. ¿Qué conserva una sesión?

Tres mensajes: dos en la misma sesión y uno en una sesión nueva.

> ✋ **Predice antes de ejecutar.** En la misma sesión, ¿atenderá los dos mensajes la misma microVM? ¿Recordará el nombre de Laura en el segundo? ¿Y en la sesión nueva?

In [ ]:
sid = new_session_id()
invocar("Me llamo Laura", session_id=sid);   m1 = ultima_microvm()
invocar("¿Cómo me llamo?", session_id=sid);  m2 = ultima_microvm(); segunda = ultima_respuesta()
invocar("¿Cómo me llamo?");                  m3 = ultima_microvm()      # sesión nueva

In [ ]:
print(f"Misma sesión → misma microVM: {m1['id'] == m2['id']}  (invocación nº {m1['invocacion']} → nº {m2['invocacion']})")
print(f"Sesión nueva → otra microVM:  {m3['id'] != m1['id']}  (invocación nº {m3['invocacion']})")
print("¿Recordó el nombre en el segundo mensaje?", "laura" in segunda.lower())

Lo esperable es: **misma microVM** (el contador sube de 1 a 2, el proceso sigue vivo con su memoria) y, aun así, **no recuerda el nombre**.

No es Runtime quien olvida. Runtime ha mantenido la microVM de la sesión con todo su estado; quien descarta la conversación es **nuestro código**, que crea un `Agent` nuevo en cada invocación (para no repetir la fuga del experimento 1). La ventana de contexto solo contiene lo que enviamos en esa llamada.

> Si la invocación devuelve `AccessDenied` sobre `InvokeModel`, el rol de ejecución del Runtime no tiene permisos sobre el inference profile y sus modelos de destino (doc §2.4).

> ✋ **Predice antes del capítulo 4.** Si la microVM conserva el estado de la sesión, ¿por qué no guardar ahí el `Agent` de cada sesión, en un diccionario? Apunta qué podría salir mal. Lo probaremos.

In [ ]:
cli('agentcore logs --since 15m')

## Para pensar

- ¿Qué diferencia hay entre *sesión de Runtime* (una microVM aislada, que vive mientras se usa) y *memoria conversacional* (estado que debe sobrevivir a la microVM)?
- ¿Qué ganas si el código del agente no depende de lo que haya en la memoria de su proceso?

## 🔎 Lo que acabamos de descubrir

Hemos aislado a los clientes… y el agente **olvida tu nombre de un mensaje al siguiente**, aunque la microVM sigue viva: nuestro código no guarda la conversación en ningún sitio. Además, cualquiera con credenciales de AWS puede invocarlo: no hay inicio de sesión de clientes.

In [ ]:
deuda("D7", "abierta", 3, "invocación con credenciales AWS, sin login de cliente")
cuaderno()
taximetro_cerrar(3)
guardar_partida(3)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap03-runtime.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap03-runtime.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `04_memoria_corto_plazo.ipynb` — *El agente que olvida tu nombre*

---
**Siguiente:** ➡️ [Capítulo 4 · El agente que olvida tu nombre](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/04_memoria_corto_plazo.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap04). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.